<center><h1>Timmala_Karan_HW3</h1></center>
<br>
<br>

Name: Karan Timmala
<br>
Github Username: karantimmala
<br>
USC ID: 1373267232

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Download the AReM data from: https://archive.ics.uci.edu/ml/datasets/
Activity+Recognition+system+based+on+Multisensor+data+fusion+\%28AReM\
%29 . The dataset contains 7 folders that represent seven types of activities. In
each folder, there are multiple files each of which represents an instant of a human
performing an activity.1 Each file containis 6 time series collected from activities
of the same person, which are called avg rss12, var rss12, avg rss13, var rss13,
vg rss23, and ar rss23. There are 88 instances in the dataset, each of which contains 6 time series and each time series has 480 consecutive values.

Package imports

Get the AReM Data Set

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import os
from pathlib import Path
import re
import zipfile
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import resample

### (b) Test and Train Data

Keep datasets 1 and 2 in folders bending1 and bending 2, as well as datasets 1,
2, and 3 in other folders as test data and other datasets as train data.

In [2]:
# =====================================================================
# DIRECTORY VERIFICATION & FILE PARSING SETUP
# =====================================================================

# Define the root path where the unzipped AReM dataset is located on your local machine
root_dir = Path(r"C:\Users\karan\OneDrive\Desktop\AReM")

# Check if the root directory actually exists before proceeding
if not root_dir.exists():
    raise FileNotFoundError(f"Directory {root_dir} was not found.")
print("Directory Verification Successful!!")

# Define the 7 target categories/activity folders mandatory for this dataset benchmark
target_categories = {"bending1","bending2","cycling","lying","sitting","standing","walking"}

# Automatically scan subfolders recursively using rglob to locate where the activity folders reside
for match_item in root_dir.rglob("bending1"):
    base_folder = match_item.parent
    
    # Collect all subdirectory names present inside the candidate parent directory
    found_subdirs = {
        sub_elem.name
        for sub_elem in base_folder.iterdir()
        if sub_elem.is_dir()
    }
    
    # If all 7 target activity folders are present in this directory, set it as dataset_root and stop searching
    if target_categories.issubset(found_subdirs):
        dataset_root = base_folder
        break

# Define ordered lists for activities and columns for standardizing data frames
activity_list = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
feature_columns = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]


def extract_index(path_obj):
    """Extracts the dataset number integer from the filename (e.g., dataset1.csv -> 1)."""
    search_res = re.search(r"dataset(\d+)", path_obj.stem)
    if search_res is None:
        raise ValueError(f"Unrecognized file naming: {path_obj.name}")
    return int(search_res.group(1))


def determine_partition(act_name, idx_num):
    """Assigns train/test split based on benchmark rules (datasets 1-2 for bending, 1-3 for others)."""
    # Bending activities use datasets 1 and 2 for testing, and the rest for training
    if act_name in ["bending1", "bending2"]:
        if idx_num in [1, 2]:
            return "test"
        return "train"
    
    # All other activities use datasets 1, 2, and 3 for testing, and the rest for training
    if idx_num in [1, 2, 3]:
        return "test"
    return "train"


def load_sensor_file(file_path):
    """Reads raw CSV time-series sensor data, cleans text lines, handles missing tokens, and performs linear interpolation."""
    collected_rows = []
    
    # Open and read the text file line by line
    with open(file_path, "r") as reader:
        for single_line in reader:
            cleaned_line = single_line.strip()
            
            # Skip empty lines or comment lines starting with '#'
            if not cleaned_line or cleaned_line.startswith("#"):
                continue
                
            # Remove trailing commas that might cause parsing artifacts
            cleaned_line = cleaned_line.rstrip(",")
            
            # Split line elements by comma and/or arbitrary whitespace
            tokens = re.split(r"\s*,\s*|\s+", cleaned_line)
            
            # Keep only lines that contain the exact expected number of sensor columns (7)
            if len(tokens) == 7:
                collected_rows.append(tokens)

    # Convert the parsed list of rows into a structured Pandas DataFrame
    df_frame = pd.DataFrame(
        collected_rows,
        columns=feature_columns
    )
    
    # Force data types to numeric, converting unparseable characters to NaN values
    df_frame = df_frame.apply(
        pd.to_numeric,
        errors="coerce"
    )
    
    # Clean up rows containing NaN values and remove duplicate timestamps
    df_frame = df_frame.dropna()
    df_frame = df_frame.drop_duplicates(subset="time")

    # Standardize time steps uniformly across time-series instances (0ms to 120,000ms at 250ms intervals)
    target_time_axis = np.arange(
        start=0,
        stop=120000,
        step=250
    )
    
    # Reindex the DataFrame to match the uniform time axis and handle any gaps using linear interpolation
    df_frame = df_frame.set_index("time")
    df_frame = df_frame.reindex(target_time_axis)
    
    sig_cols = feature_columns[1:]
    df_frame[sig_cols] = df_frame[sig_cols].interpolate(
        method="linear",
        limit_direction="both"
    )
    
    # Reset index back to a standard column format
    df_frame.index.name = "time"
    df_frame = df_frame.reset_index()
    return df_frame
    

# Initialize a tracking list to store metadata records for every CSV instance
log_entries = []

# Loop iteratively through each activity folder and parse each matching dataset CSV file
for act in activity_list:
    target_dir = dataset_root / act
    
    # Sort files numerically using the helper extraction function
    csv_files = sorted(
        target_dir.glob("dataset*.csv"),
        key=extract_index
    )
    for current_file in csv_files:
        file_idx = extract_index(current_file) 
        # Load, clean, and interpolate the time-series file
        processed_df = load_sensor_file(current_file)
        
        # Append metadata and record information for tracking/validation summary
        log_entries.append({
            "activity": act,
            "dataset": file_idx,
            "split": determine_partition(act, file_idx),
            "observations": len(processed_df),
            "file": current_file
        })

# Compile the audit records into a summary DataFrame to confirm correct instance count and splits
summary_validation = pd.DataFrame(log_entries)

print("TOTAL INSTANCES:", len(summary_validation))
print("---------------------------------")
print(summary_validation["activity"].value_counts())
print("---------------------------------")
print(summary_validation["split"].value_counts())
print("---------------------------------")
print(summary_validation["observations"].value_counts())

Directory Verification Successful!!
TOTAL INSTANCES: 88
---------------------------------
activity
cycling     15
lying       15
sitting     15
standing    15
walking     15
bending1     7
bending2     6
Name: count, dtype: int64
---------------------------------
split
train    69
test     19
Name: count, dtype: int64
---------------------------------
observations
480    88
Name: count, dtype: int64


### (c) Feature Extraction

#### i. Research what types of time-domain features are usually used in time series classification and list them (examples are minimum, maximum, mean, etc).


Min & Max: Mark the lower and upper boundaries of the signal amplitude.

Mean & Median: Represent central trend metrics to establish the signal's baseline level.

Standard Deviation & Variance: Quantify variability and fluctuation in signal energy or amplitude.

Percentiles / Interquartile Range (IQR): Measure data spread at given thresholds (e.g., 25th and 75th percentiles), offering outlier-resistant dispersion metrics.

Skewness: Gauges the degree of asymmetry in the data distribution around the mean.

Kurtosis: Indicates the presence of heavy tails or extreme outlier values in the signal.

Mean Absolute Deviation (MAD): Average distance of points from the mean.

Root Mean Square (RMS): Quantifies the overall power/magnitude of the signal.

Entropy: Quantifies the degree of complexity, regularity, or randomness in the sequence.



#### ii. Extract the time-domain features minimum, maximum, mean, median, standard deviation, first quartile, and third quartile for all of the 6 time series in each instance. You are free to normalize/standardize features or use them directly.


In [3]:
# =====================================================================
# FEATURE EXTRACTION (TIME-DOMAIN METRICS)
# =====================================================================

def compute_summary_metrics(df_source):
    """
    Computes time-domain statistical metrics (min, max, mean, median, standard deviation, 
    1st quartile, and 3rd quartile) for each of the 6 sensor time-series channels.
    """
    metrics_dict = {}
    
    # Exclude the first column ('time') and process only the 6 sensor columns
    sig_fields = feature_columns[1:]

    # Iterate through each sensor column, assigning a numerical index starting from 1
    for idx_num, col_label in enumerate(sig_fields, start=1):
        series_data = df_source[col_label]
        
        # Calculate time-domain summary statistics for the current sensor channel
        metrics_dict[f"min{idx_num}"] = series_data.min()
        metrics_dict[f"max{idx_num}"] = series_data.max()
        metrics_dict[f"mean{idx_num}"] = series_data.mean()
        metrics_dict[f"median{idx_num}"] = series_data.median()
        metrics_dict[f"std{idx_num}"] = series_data.std(ddof=1)  # Sample standard deviation (N-1 degrees of freedom)
        metrics_dict[f"1st_quart{idx_num}"] = series_data.quantile(0.25)  # 25th percentile
        metrics_dict[f"3rd_quart{idx_num}"] = series_data.quantile(0.75)  # 75th percentile
        
    return metrics_dict


# Initialize a list to hold the computed feature dictionaries for all dataset instances
record_collection = []

# Loop through each activity folder and file to extract summary features for all 88 instances
for act_name in activity_list:
    target_dir = dataset_root / act_name
    
    # Retrieve and numerically sort all dataset CSV files in the current activity directory
    csv_collection = sorted(
        target_dir.glob("dataset*.csv"),
        key=extract_index
    )
    
    for single_csv in csv_collection:
        # Extract metadata identifiers for the current file
        file_id = extract_index(single_csv)
        
        # Load, clean, and interpolate the time-series sensor data
        processed_frame = load_sensor_file(single_csv)
        
        # Compute the time-domain metrics dictionary for all 6 sensor channels
        stat_summary = compute_summary_metrics(processed_frame)
        
        # Append instance identification metadata to the summary dictionary
        stat_summary["instance"] = f"{act_name}_dataset{file_id}"
        stat_summary["activity"] = act_name
        stat_summary["dataset"] = file_id
        stat_summary["split"] = determine_partition(act_name, file_id)
        
        # Store the completed dictionary record in the collection list
        record_collection.append(stat_summary)

# Convert the compiled list of records into a comprehensive pandas DataFrame
extracted_features = pd.DataFrame(record_collection)

# Define the sequence of statistical indicators to build structured feature column names
stat_indicators = [
    "min",
    "max",
    "mean",
    "median",
    "std",
    "1st_quart",
    "3rd_quart"
]

# Generate the complete, ordered list of 42 feature column names (7 metrics x 6 channels)
generated_feature_names = []
for chan_num in range(1, 7):
    for indicator in stat_indicators:
        generated_feature_names.append(f"{indicator}{chan_num}")

# Reorder the DataFrame columns to place metadata columns first, followed by the generated sensor features
extracted_features = extracted_features[
    ["instance", "activity", "dataset", "split"] + generated_feature_names
]

# Sort the final dataset rows by activity name and dataset number, then reset the index
extracted_features = extracted_features.sort_values(
    ["activity", "dataset"]
).reset_index(drop=True)

# Preview the first 5 rows of the extracted feature table
extracted_features.head()

,instance,activity,dataset,split,min1,max1,mean1,median1,std1,1st_quart1,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1_dataset1,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1_dataset2,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1_dataset3,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1_dataset4,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1_dataset5,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


In [4]:
# MATRIX DIMENSIONS & MISSING VALUE CHECK
matrix_view = extracted_features[
    ["instance"] + generated_feature_names
]
print("TABLE DIMENSIONS", matrix_view.shape)
matrix_view

TABLE DIMENSIONS (88, 43)


,instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1_dataset1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,bending1_dataset2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,bending1_dataset3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,bending1_dataset4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,bending1_dataset5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,walking_dataset11,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,walking_dataset12,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,walking_dataset13,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,walking_dataset14,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,14.82,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


In [5]:
# BOOTSTRAP CONFIDENCE INTERVAL ESTIMATION
extracted_features["split"].value_counts()
null_tally = extracted_features[generated_feature_names].isna().sum().sum()
print("MISSING VALUE COUNT:", null_tally)

MISSING VALUE COUNT: 0


#### iii. Standard Deviation Estimate the standard deviation of each of the time-domain features you extracted from the data. Then, use Python’s bootstrapped or any other method to build a 90% bootsrap confidence interval for the standard deviation of each feature.

In [6]:
def compute_bootstrap_bounds(arr_data, total_samples=10000, init_seed=42):
    """Calculates a 90% bootstrap confidence interval for standard deviation using empirical resampling."""
    rnd_engine = np.random.default_rng(init_seed)
    stat_array = np.empty(total_samples)
    for idx in range(total_samples):
        resample_subset = rnd_engine.choice(
            arr_data,
            size=len(arr_data),
            replace=True
        )
        stat_array[idx] = np.std(resample_subset, ddof=1)
    lower_limit = np.percentile(stat_array, 5)
    upper_limit = np.percentile(stat_array, 95)
    return lower_limit, upper_limit

ci_records = []

# Generate bootstrap intervals for each extracted time-domain feature
for pos_idx, col_name in enumerate(generated_feature_names):
    column_array = extracted_features[col_name].to_numpy()
    sample_std_dev = np.std(column_array, ddof=1)
    lower_val, upper_val = compute_bootstrap_bounds(
        column_array,
        total_samples=10000,
        init_seed=42 + pos_idx
    )
    ci_records.append({
        "FEATURE": col_name,
        "STANDARD DEVIATION": sample_std_dev,
        "LOWER BOUND": lower_val,
        "UPPER BOUND": upper_val
    })
bootstrap_summary_df = pd.DataFrame(ci_records)
bootstrap_summary_df = bootstrap_summary_df.round(3)
bootstrap_summary_df

,FEATURE,STANDARD DEVIATION,LOWER BOUND,UPPER BOUND
0,min1,9.570,8.236,10.774
1,max1,4.394,3.345,5.290
2,mean1,5.336,4.711,5.877
3,median1,5.440,4.796,5.993
4,std1,1.772,1.573,1.950
5,1st_quart1,6.154,5.574,6.643
6,3rd_quart1,5.139,4.336,5.830
7,min2,0.000,0.000,0.000
8,max2,5.063,4.622,5.408
9,mean2,1.574,1.396,1.706


#### iv. Use your judgement to select the three most important time-domain features (one option may be min, mean, and max).

##### Based on my judgement, I selected Mean, Maximum and Standard Deviation as the three most important time-domain features.
##### **Maximum:** The maximum value records the highest intensity peak within a given time frame. Even when different activities share similar average values, their peak outputs often differ sharply, allowing this feature to highlight sudden, high-intensity movements.
##### **Mean:** The mean establishes the signal's core baseline over time. Because distinct physical activities typically function at varying baseline intensities, this metric makes it easy to separate actions with sustained high intensity from those with lower overall activity levels.
##### **Standard Deviation:** Standard deviation measures the degree of variation and instability within the signal. Continuous physical movement generally causes greater signal fluctuation, while stationary positions keep the signal stable, making variability a key metric for distinguishing between active and idle states.

##### Together, these three parameters capture central tendency, variance, and peak values, providing a concise yet effective feature set to differentiate time-series signal behaviors. 

# 2. ISLR 3.7.4

### I collect a set of data (n = 100 observations) containing a single predictor and a quantitative response. I then fit a linear regression model to the data, as well as a separate cubic regression, i.e. Y = β0 + β1X + β2X2 + β3X3 + ϵ.


### (a) Suppose that the true relationship between X and Y is linear, i.e. Y = β0 + β1X + ϵ. Consider the training residual sum of squares (RSS) for the linear regression, and also the training RSS for the cubic regression. Would we expect one to be lower than the other, would we expect them to be the same, or is there not enough information to tell? Justify your answer.

##### Ans : When the true underlying relationship between $X$ and $Y$ is linear, a linear regression model accurately captures the underlying process. Nevertheless, a cubic regression model inherently encompasses the linear model when its higher-order coefficients ($\beta_2$ and $\beta_3$) equal zero. Because of these extra parameters, the cubic model possesses greater capacity to fit the noise in the training set. Consequently, its training residual sum of squares (RSS) will always be lower than or equal to that of the linear model, as the increased flexibility allows it to fit the training data more tightly even if the actual pattern is linear.

### **(b) Answer (a) using test rather than training RSS.**


##### Ans : When evaluating test RSS, the outcome shifts in favor of the simpler model. Because the actual data-generating process is linear, the linear regression achieves minimal bias without introducing redundant complexity. Conversely, the cubic regression introduces extra degrees of freedom, which inflates the model's variance and leaves it prone to overfitting by fitting random noise present in the training set. Consequently, the linear model is anticipated to achieve a lower test RSS compared to the cubic model.

### 
**(c) Suppose that the true relationship between X and Y is not linear, but we don’t know how far it is from linear. Consider the training RSS for the linear regression, and also the training RSS for the cubic regression. Would we expect one to be lower than the other, would we expect them to be the same, or is there not enough information to tell? Justify your answer.**

##### Ans : When the actual relationship between $X$ and $Y$ is non-linear, the cubic regression provides greater flexibility than the linear model. Since the cubic specification encompasses the linear model as a subset where $\beta_2 = \beta_3 = 0$, it can adapt to the training observations at least as effectively as the simpler model. Consequently, the cubic regression will yield a training RSS that is less than or equal to that of the linear model.

### **(d)Answer (c) using test rather than training RSS.**

##### Ans : When evaluating test RSS under non-linear conditions, there is insufficient information to conclusively determine which model will outperform the other. If the true data-generating process is only slightly non-linear, the linear regression may yield better generalization due to its minimal model variance. Conversely, if the non-linear structure is pronounced, the cubic regression's ability to significantly lower estimation bias could outweigh its higher variance, resulting in a superior test RSS. Consequently, relative performance depends on the underlying bias–variance tradeoff.

### AI USED

1.How can a 90% percentile-based bootstrap confidence interval for a feature's standard deviation be implemented programmatically in Python?

2.What are the best practices for managing missing values ($NaNs$) when calculating sample standard deviations and bootstrap estimates?

3.What is the standard Python approach for calculating summary statistics (min, max, mean, median, standard deviation, and the 25th and 75th percentiles) across sensor channels?

4.What are the standard time-domain statistical indicators used when feature-engineering time-series classification models?

### REFERENCES

https://nerdnodes2023.medium.com/feature-engineering-for-time-series-classification-a-comprehensive-guide-f2073084718a

https://machinelearningmastery.com/feature-selection-time-series-forecasting-python/

https://machinelearningmastery.com/how-to-calculate-the-5-number-summary-for-your-data-in-python/

https://statisticsfundamentals.com/sampling-distributions/bootstrap-sampling/

https://www.dasca.org/world-of-data-science/article/strategies-for-handling-missing-values-in-data-analysis